# ============================================================
# STEP 6 — FLAML AutoML
# Turkish Quishing Detection v10
# ============================================================
# Runs Microsoft FLAML AutoML to validate manual model choices.
# If FLAML's auto-tuned model ≈ our manual XGBoost, it confirms
# our configuration was near-optimal. If it beats you, adopt it.
#
# Output: step6_flaml_best.pkl, step6_flaml_vs_manual.csv
# ============================================================

In [ ]:

import os, time, pickle, warnings, json
import numpy as np
import pandas as pd
from sklearn.metrics import (
    roc_auc_score, f1_score, recall_score, matthews_corrcoef)

warnings.filterwarnings("ignore")
SEED = 42

DATA_DIR = "/content/drive/MyDrive/TUMC_dataset"
INPUT_FILE  = os.path.join(DATA_DIR, "features_full_final_inductive_dedup.csv")
OUT_PKL     = os.path.join(DATA_DIR, "step6_flaml_best.pkl")
OUT_CSV     = os.path.join(DATA_DIR, "step6_flaml_vs_manual.csv")
METRICS_CSV = os.path.join(DATA_DIR, "step3_all_metrics_final.csv")

TASK          = "binary"
TIME_BUDGET   = 120       # seconds for AutoML search
DROP_LEAKAGE  = True      # use honest variant (no is_tr_domain/is_https)

try:
    from flaml import AutoML
except ImportError:
    print("Run: !pip install flaml -q"); raise SystemExit

print("=" * 70)
print(f"STEP 6 — FLAML AutoML ({TASK}, {TIME_BUDGET}s budget)")
print("=" * 70)

df = pd.read_csv(INPUT_FILE, low_memory=False)
META = {"url","source","tld","label","label_enc",
        "class_final","class_enc","fold","reg_domain"}
FEATURES = [c for c in df.columns if c not in META]
if DROP_LEAKAGE:
    FEATURES = [f for f in FEATURES if f not in ("is_tr_domain","is_https")]
    print(f"  Honest variant: {len(FEATURES)} features (leakage removed)")

target_col = "label_enc" if TASK == "binary" else "class_enc"
is_binary  = (TASK == "binary")

# Domain-aware split: folds 1-4 train, fold 0 test
tr = df[df["fold"] != 0]
te = df[df["fold"] == 0]
X_tr, y_tr = tr[FEATURES], tr[target_col]
X_te, y_te = te[FEATURES], te[target_col]
print(f"  Train: {len(X_tr):,}  Test: {len(X_te):,}")

# ── Run FLAML ─────────────────────────────────────────────────
print(f"\n[1] Running AutoML search ({TIME_BUDGET}s) ...")
automl = AutoML()
settings = {
    "time_budget": TIME_BUDGET,
    "metric": "roc_auc" if is_binary else "macro_f1",
    "task": "classification",
    "estimator_list": ["lgbm", "xgboost", "rf", "extra_tree"],
    "seed": SEED,
    "eval_method": "cv",
    "n_splits": 3,
    "verbose": 1,
}
t0 = time.time()
automl.fit(X_train=X_tr, y_train=y_tr, **settings)
elapsed = time.time() - t0

print(f"\n[2] Best config found in {elapsed:.0f}s:")
print(f"    Estimator: {automl.best_estimator}")
print(f"    Config:    {automl.best_config}")

# ── Evaluate on held-out fold 0 ───────────────────────────────
pred  = automl.predict(X_te)
proba = automl.predict_proba(X_te)
m = {"mcc": matthews_corrcoef(y_te, pred)}
if is_binary:
    m["auc"]    = roc_auc_score(y_te, proba[:, 1])
    m["recall"] = recall_score(y_te, pred, zero_division=0)
    m["f1"]     = f1_score(y_te, pred, zero_division=0)
else:
    m["auc_ovr"]  = roc_auc_score(y_te, proba, multi_class="ovr", average="macro")
    m["f1_macro"] = f1_score(y_te, pred, average="macro", zero_division=0)

print(f"\n[3] FLAML test performance:")
for k, v in m.items():
    print(f"    {k:<12s}: {v:.4f}")

# ── Compare with manual models from Step 3 ───────────────────
print(f"\n[4] FLAML vs manual models (Step 3):")
comparison = [{"source": "FLAML_AutoML",
               "estimator": automl.best_estimator, **m}]
if os.path.exists(METRICS_CSV):
    man = pd.read_csv(METRICS_CSV)
    man = man[(man["task"]==TASK)]
    key = "auc" if is_binary else "f1_macro"
    if key in man.columns and len(man):
        best_manual = man.nlargest(1, key).iloc[0]
        print(f"    Best manual: {best_manual['model']} "
              f"({key}={best_manual[key]:.4f})")
        print(f"    FLAML:       {automl.best_estimator} "
              f"({key}={m.get(key,0):.4f})")
        diff = m.get(key,0) - best_manual[key]
        verdict = "FLAML wins" if diff > 0.001 else \
                  "manual wins" if diff < -0.001 else "tie"
        print(f"    Δ = {diff:+.4f} → {verdict}")
        comparison.append({"source": f"manual_{best_manual['model']}",
                           "estimator": best_manual["model"],
                           key: best_manual[key],
                           "mcc": best_manual.get("mcc", np.nan)})

pd.DataFrame(comparison).to_csv(OUT_CSV, index=False)
with open(OUT_PKL, "wb") as f:
    pickle.dump(automl, f)

print(f"""
{'='*70}
STEP 6 COMPLETE
{'='*70}
  Best estimator : {automl.best_estimator}
  Saved          : {OUT_PKL}
  Comparison     : {OUT_CSV}

  Interpretation:
    FLAML ≈ manual → our manual tuning was near-optimal (good)
    FLAML > manual → adopt FLAML config for final model
  Next → Step 7: CNN + BiLSTM (raw URL deep learning)
{'='*70}
""")

STEP 6 — FLAML AutoML (binary, 120s budget)
  Honest variant: 137 features (leakage removed)
  Train: 991,213  Test: 248,095

[1] Running AutoML search (120s) ...

[2] Best config found in 127s:
    Estimator: lgbm
    Config:    {'n_estimators': 70, 'num_leaves': 11, 'min_child_samples': 11, 'learning_rate': np.float64(0.14244398303666542), 'log_max_bin': 6, 'colsample_bytree': np.float64(0.996498378007316), 'reg_alpha': np.float64(0.002255250947893723), 'reg_lambda': np.float64(2.244815726898779)}

[3] FLAML test performance:
    mcc         : 0.9650
    auc         : 0.9990
    recall      : 0.9933
    f1          : 0.9948

[4] FLAML vs manual models (Step 3):

STEP 6 COMPLETE
  Best estimator : lgbm
  Saved          : /content/drive/MyDrive/Colab Notebooks/Research/New_Dataset_May/step6_flaml_best.pkl
  Comparison     : /content/drive/MyDrive/Colab Notebooks/Research/New_Dataset_May/step6_flaml_vs_manual.csv

  Interpretation:
    FLAML ≈ manual → your manual tuning was near-optima